# CMSC 173 &middot; Machine Learning &mdash; Week 9 Lab
## Logistic Regression from Scratch

Linear regression predicts a number; **logistic regression** predicts a *probability* &mdash; is this
email spam, is this patient positive? You'll build it from the ground up: the **sigmoid** that
turns a score into a probability, the **cross-entropy** loss, and **gradient descent** to train &mdash;
then draw the **decision boundary** it learns.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib (from scratch).** **Not graded.** About 55 minutes.

---
## Part 0 &middot; Setup + two classes to separate

Two clouds of points in 2-D &mdash; say students who passed (1) vs failed (0), from two exam scores.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(173)

n = 200
y = rng.integers(0, 2, n)
X = rng.normal(np.c_[np.where(y==1, 2.0, -0.3), np.where(y==1, 1.5, -0.3)], 1.1)

plt.figure(figsize=(6,5))
plt.scatter(X[y==0,0], X[y==0,1], marker='o', alpha=0.6, label='class 0 (fail)')
plt.scatter(X[y==1,0], X[y==1,1], marker='^', alpha=0.6, label='class 1 (pass)')
plt.xlabel('exam A'); plt.ylabel('exam B'); plt.legend(); plt.title('Can we draw a line between them?')
plt.tight_layout(); plt.show()

**Reading the code:** class 1 sits up-and-right of class 0, with overlap. Our model will learn a
straight boundary that best separates them &mdash; and, importantly, a *probability* for points near it.

---
## Part 1 &middot; The sigmoid: score &rarr; probability

A linear score $z = \theta^\top x$ can be any number. The **sigmoid** squashes it into $(0,1)$ so we
can read it as a probability: $\sigma(z) = 1/(1+e^{-z})$. Big positive $z\to$ near 1, big negative
$\to$ near 0, and $\sigma(0)=0.5$.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

zs = np.linspace(-8, 8, 200)
plt.figure(figsize=(6,3.5)); plt.plot(zs, sigmoid(zs))
plt.axhline(0.5, ls='--', color='gray'); plt.axvline(0, ls='--', color='gray')
plt.xlabel('score z'); plt.ylabel('sigmoid(z) = probability'); plt.title('The sigmoid')
plt.tight_layout(); plt.show()
print('sigmoid(0) =', sigmoid(0), '| sigmoid(4) =', round(sigmoid(4),3), '| sigmoid(-4) =', round(sigmoid(-4),3))

**Reading the code:** `sigmoid` is one line. The S-shaped curve maps any score onto a probability;
the dashed lines show the tipping point at `z=0` where the probability is exactly 0.5. A score of
+4 is ~98% class 1; &minus;4 is ~2%.

**Answer here:**

1. Why can't we just use the raw linear score as a probability &mdash; what's wrong with a 'probability'
   of 7.3 or &minus;2?
   &rarr; *your answer*

---
## Part 2 &middot; The loss: cross-entropy

How wrong is a probability? **Cross-entropy** punishes confident-and-wrong predictions harshly:
if the true label is 1 it uses $-\log(\hat p)$, if 0 it uses $-\log(1-\hat p)$. Predict 0.99 when the
truth is 0 and the loss explodes. (This is why we don't reuse MSE here.)

In [ ]:
def loss(y, p):
    p = np.clip(p, 1e-9, 1-1e-9)                      # avoid log(0)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

print('loss if we guess 0.5 for everyone :', round(loss(y, np.full(n, 0.5)), 3))
print('loss for a confident WRONG guess  :', round(loss(np.array([1]), np.array([0.01])), 3))

**Reading the code:** the two-term formula is the boxed one; `np.clip` keeps probabilities away from
exactly 0 or 1 so `log` never blows up. Guessing 0.5 for everyone gives a moderate loss (~0.69);
being 99% sure of the wrong answer gives a huge one &mdash; the model is strongly pushed to avoid that.

---
## Part 3 &middot; Train it with gradient descent

Put it together: predict with the sigmoid, measure with cross-entropy, and step downhill. The
gradient has a beautifully simple form &mdash; the *same* as linear regression's:

$$\nabla = \tfrac{1}{m} X^\top(\hat p - y).$$

In [ ]:
Xb = np.column_stack([np.ones(n), X])                # (1) add a 1s column for the bias
theta = np.zeros(3)                                  # (2) start at all-zero
lr, iters = 0.1, 2000
history = []
for _ in range(iters):
    p = sigmoid(Xb @ theta)                           # (3) current probabilities
    grad = Xb.T @ (p - y) / n                         # (4) the gradient
    theta -= lr * grad                                # (5) step downhill
    history.append(loss(y, p))                        # (6) record the loss

print('learned theta:', np.round(theta, 2))
plt.figure(figsize=(6,3.5)); plt.plot(history)
plt.xlabel('iteration'); plt.ylabel('cross-entropy loss'); plt.title('Training loss falls')
plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** prepend a column of 1s so `theta[0]` is the bias/intercept.
- **(2)** start every parameter at zero.
- **(3)** `sigmoid(Xb @ theta)` gives each point's current probability of class 1.
- **(4)** the gradient `Xᵀ(p - y)/m` &mdash; identical shape to linear regression's, just with `p` from
  the sigmoid.
- **(5)** step against the gradient.
- **(6)** the loss curve slides down and flattens &mdash; training is working.

**Answer here:**

1. The gradient here looks exactly like linear regression's `Xᵀ(pred - y)/m`. Why is that a nice
   thing (think about reusing code / intuition)?
   &rarr; *your answer*

---
## Part 4 &middot; The decision boundary it learned

The model predicts class 1 when its probability passes 0.5, i.e. where the score $\theta^\top x = 0$.
That's a straight line in our 2-D space &mdash; let's draw it over the data, shaded by probability.

In [ ]:
# grid of points to colour by predicted probability
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200),
                     np.linspace(X[:,1].min()-1, X[:,1].max()+1, 200))
grid = np.column_stack([np.ones(xx.size), xx.ravel(), yy.ravel()])
probs = sigmoid(grid @ theta).reshape(xx.shape)

plt.figure(figsize=(6.5,5))
plt.contourf(xx, yy, probs, levels=20, cmap='RdBu_r', alpha=0.6)   # probability shading
plt.contour(xx, yy, probs, levels=[0.5], colors='k')              # the boundary at p=0.5
plt.scatter(X[y==0,0], X[y==0,1], marker='o', edgecolor='k', label='class 0')
plt.scatter(X[y==1,0], X[y==1,1], marker='^', edgecolor='k', label='class 1')
plt.legend(); plt.title('Learned decision boundary (black line = 50% probability)')
plt.tight_layout(); plt.show()

**Reading the code:** we score a fine grid of points, shade them by predicted probability (blue&rarr;red
= class 0&rarr;1), and draw the single black contour where the probability is 0.5 &mdash; the **decision
boundary**. Notice the shading is *soft* near the line: the model is honestly unsure about the
overlapping points, which is the whole advantage of predicting probabilities.

**Answer here:**

1. Some points sit on the wrong side of the boundary. Given the two clouds overlap, is that a bug
   or expected? What would perfectly separating them risk (week-4 word)?
   &rarr; *your answer*

---
## Part 5 &middot; Predictions & accuracy

Turn probabilities into 0/1 labels at the 0.5 threshold and score the model.

In [ ]:
pred = (sigmoid(Xb @ theta) >= 0.5).astype(int)
accuracy = np.mean(pred == y)
print(f'training accuracy = {accuracy:.3f}')

**Answer here:**

1. This is *training* accuracy. From weeks 6-7, what should you really do before trusting this
   number?
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| What the sigmoid does | - |
| Why cross-entropy, not MSE | - |
| The gradient-descent training loop | - |
| Reading a decision boundary | - |
| Turning probabilities into labels | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: how is logistic regression different from linear regression?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; Check against sklearn

`from sklearn.linear_model import LogisticRegression` &mdash; fit it on `X, y` and compare its accuracy
to yours. Fill it in.

In [ ]:
# from sklearn.linear_model import LogisticRegression
# your code here: fit LogisticRegression().fit(X, y) and print its .score(X, y) next to yours


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 9

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/9/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 9 submission page](https://portal.latarak.com/course/cmsc173/lab/9/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.